### **Cyclic ID Algorithm Tutorial**
---

In this tutorial notebook, you'll see how to use the cyclic ID algorithm (Forré & Mooij, 2019) to identify causal queries in cyclic graphs that contain:

- Feedback Loops
- Latent confounders (unmeasured causes)
- Both of these simultaneously (consolidated districts)

We demonstrate the algorithm across four scenarios:

1. Cyclic graph with an identifiable query
2. Cyclic graph with a non-identifiable query
3. Acyclic graph with an identifiable query
4. Acyclic graph with a non-identifiable query




### **Why use the Cyclic ID Algorithm?**

Traditional causal inference algorithms like the ID algorithm assume graphs are acyclic (no feedback loops). However, many real systems contain cycles. For example, in biological systems, gene regulatory networks regulate each other which can cause many complex feedback loops and latent confounders.

### The Challenge

When cycles are present in a graph, the traditional methods fail because:

1. There is no topological ordering of variables
2. Standard conditional independence relationships break down
3. Feedback loops create complex dependencies

In [3]:
# importing libraries from y0

from y0.dsl import A, B, C, D, P, E, X, Y, Z, R, W, V1, Z1, Z2 
from y0.graph import NxMixedGraph
from y0.algorithm.identify.cyclic_id import cyclic_id, Unidentifiable
from y0.examples import napkin

print("Imports successful.")

Imports successful.


### Scenario 1 & 2: Cyclic Graph

For the first two examples, we will use a cyclic graph with the following features:

1. **A feedback loop**: B → C → D → B (nodes affect each other in a cycle)
2. **A self-loop**: E → E (a node that feeds back to itself)
3. **A latent confounder**: B ↔ D (an unmeasured common cause shown as a bidirected edge) 

These features create a **consolidated district** which is a group of nodes tightly connected 
through cycles and confounders. The Cyclic ID algorithm is designed to handle these kinds of cases.

The graph structure is:
- A → B, A → E
- B → C → D → B (feedback loop)
- C → E
- E → E (self-loop)
- B ↔ D (latent confounder)

In [2]:
# cyclic graph example for scenario 1 and 2
graph = NxMixedGraph.from_edges(
    directed=[
        (A, B),    # A -> B
        (B, C),    # B -> C 
        (C, D),    # C -> D
        (D, B),    # D -> B: completes feedback loop 
        (C, E),    # C -> E
        (A, E),    # A -> E
        (E, E),    # E -> E: self-loop
    ],
    undirected=[
        (B, D),    # B <-> D: latent confounder
    ]
)

print("graph constructed")
print(f"Nodes: {sorted(graph.nodes(), key=str)}")
print(f"Directed edges: {len(graph.directed.edges())} edges")
print(f"Bidirected edges: {len(graph.undirected.edges())} bidirected edge(s)")

graph constructed
Nodes: [A, B, C, D, E]
Directed edges: 7 edges
Bidirected edges: 1 bidirected edge(s)


### Scenario 1: Cyclic Graph — Identifiable Query

For the first scenario we ask: **what is the causal effect of A on E?**

In causal inference notation: **P(E | do(A))**

### Why might this be identifiable?

- **A is external to the feedback loop** — it sits outside the consolidated district {B, C, D}
- **There are multiple paths from A to E** — A → E directly and A → B → C → E
- Intervening on A from outside the cycle allows the algorithm to trace the effect through to E

**Prediction: ✅ Identifiable**

In [5]:
# Scenario 1: Cyclic graph with identifiable query
# P(E | do(A)) — intervening from outside the feedback loop
print("=" * 60)
print("SCENARIO 1: P(E | do(A))")
print("=" * 60)

try:
    result_1 = cyclic_id(graph, outcomes={E}, interventions={A})
    print("\n✓ Result: ✅ IDENTIFIABLE")
    print("\nSymbolic expression:")
    display(result_1)
except Unidentifiable as e:
    print("\n✗ Result: ❌ UNIDENTIFIABLE")
    print(f"Reason: {e}")

SCENARIO 1: P(E | do(A))

✓ Result: ✅ IDENTIFIABLE

Symbolic expression:


Sum[B, C, D]((P(A, B, C, D, E) / P(A)))

### What does this result mean? - Scenario 1 (Identifiable Query)

### Scenario 2: Cyclic Graph - Non-Identifiable Query



In [6]:

print("=" * 60)
print("SCENARIO 2: P(D | do(B))")
print("=" * 60)

try:
    result_2 = cyclic_id(graph, outcomes={D}, interventions={B})
    print("\n✓ Result: ✅ IDENTIFIABLE")
    print("\nSymbolic expression:")
    display(result_2)
    print("\nSimplified:")
    display(result_2.simplify())
except Unidentifiable as e:
    print("\n✗ Result: ❌ UNIDENTIFIABLE")
    print(f"Reason: {e}")

SCENARIO 2: P(D | do(B))

✗ Result: ❌ UNIDENTIFIABLE
Reason: Cannot identify P{D} | do{B})). District frozenset({C}) failed identification.
